In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
tables = q("""
    select table_name
    from information_schema.tables
    where table_schema = 'prime'
    order by table_name
""")

tables

Вопрос моего исследования:
Какие факторы определили изменение выручки от подписок «Прайм» и в каких тарифах или способах оплаты находятся основные возможности ее роста? 

Задачи моего исследования 
1. Проверить как изменилась выручка между периодами
2. Посмотреть какой вклад внесли количество успешных оплат и средний платеж
3. Посмотреть какие тарифы и способы оплаты связаны с основными изменениями
4. Определить что бизнесу стоит предпринять или дополнительно проверить и написать это в выводах работы 

In [ ]:
payment_overview = q("""select
        status,
        count(*) as n_payments,
        min(paid_at) as first_payment,
        max(paid_at) as last_payment
    from prime.payments
    group by status
    order by status""")

payment_overview

#В этой ячейке я создала сводную таблицу по статусам платежей, где посчитала количество попыток оплаты для каждого статуса и определила даты первой и последней записи, что позволило узнать, за какой период доступны данные, и различить успешные, неуспешные и возвращенные платежи. Эта информация нужна для дальнейшего выбора периодов сравнения и корректного расчета выручки 

Загружаю данные о попытках оплаты подписок за первое полугодие 2026 года без ограничения персональным сегментом из домашних заданий и сохраняю все статусы и способы оплаты, чтобы далее исследовать выручку и структуру попыток оплаты (одна строка – одной попытке оплаты)

In [ ]:
pay = q("""
    select payment_id, subscription_id, paid_at,
           amount, status, payment_method
    from prime.payments
    where paid_at >= :start
      and paid_at < :end
    order by payment_id
""", start="2026-01-01", end="2026-07-01")

print("Количество строк:", len(pay))
pay.head()

Перед расчетом показателей проверяю количество загруженных записей, наличие пропусков в столбцах и повторяющихся идентификаторов платежей, чтобы выявить возможные проблемы с данными и избежать двойного счета при расчете выручки

In [ ]:
print("Количество строк:", len(pay))

print("Пропуски по столбцам:")
print(pay.isna().sum())

print("Повторяющиеся payment_id:")
print(pay["payment_id"].duplicated().sum())

#про результат выше
Пропусков и повторяющихся идентификаторов платежей нет, удаление строк не требуется

Создаю отдельную таблицу успешных оплат для расчета выручки (неуспешные попытки и возвращенные платежи в нее не входят)

In [ ]:
success = pay[pay["status"] == "success"].copy()

print("Успешных оплат:", len(success))

Теперь расчет выручки за первое полугодие 2026 года. Полученное значение будет контрольной суммой (при дальнейшем разбиении выручки по месяцам, тарифам и способам оплаты итог должен с ним совпадать) 

In [ ]:
total_revenue = success["amount"].sum()

print("Выручка за полугодие:", round(total_revenue, 2), "₽")

Рассматриваю распределение по месяцам 

In [ ]:
success["paid_at"] = pd.to_datetime(success["paid_at"], utc=True)
success["month"] = success["paid_at"].dt.strftime("%Y-%m")

success[["paid_at", "month"]].head()

In [ ]:
monthly = success.groupby("month").agg(
    revenue=("amount", "sum"),
    n_payments=("payment_id", "size"),
    avg_payment=("amount", "mean"))

monthly

In [ ]:
#проверка совпадения 
monthly_total = monthly["revenue"].sum()

difference = round(monthly_total - total_revenue, 2)
print("Расхождение:", difference, "₽")

assert difference == 0, "Контроль ДЕНЬГИ не пройден"

In [ ]:
#Проверяю наличие записей на кадоый день
pay["paid_at"] = pd.to_datetime(pay["paid_at"], utc=True)

daily_counts = pay.groupby(
    pay["paid_at"].dt.normalize()).size()

calendar = pd.date_range(
    start="2026-01-01",
    end="2026-06-30",
    freq="D",
    tz="UTC")

daily_counts = daily_counts.reindex(calendar, fill_value=0)

daily_counts[daily_counts == 0]

#Про резльтат выше 
При проверке календарного покрытия я обнаружила, что за 14 и 15 апреля 2026 года отсутствуют записи о попытках оплаты всех статусов. Однако по имеющимся данным нельзя установить, связана ли эта ситуация с отсутствием операций или неполнотой данных, поэтому апрель исключаю из основного сравнения полных месяцев, а рассчитанную апрельскую выручку рассматриваю как сумму по доступным записям 

In [ ]:
#добавила число календарных дней 
monthly["calendar_days"] = pd.to_datetime(
    monthly.index + "-01").days_in_month

monthly[["revenue", "calendar_days"]]

Для сравнения месяцев разной продолжительности рассчитываю среднюю выручку на календарный день (апрель исключаю из сравнительной таблицы из-за отсутствия записей за два дня). Исходную помесячную сводку сохраняю без удаления строк

In [ ]:
monthly_comparison = monthly.drop(index="2026-04").copy()

monthly_comparison["revenue_per_day"] = (
    monthly_comparison["revenue"] / monthly_comparison["calendar_days"])

monthly_comparison[
    ["revenue", "calendar_days", "revenue_per_day"]].round(2)

В феврале месячная выручка снизилась с 46.91млн до 43.50млн, однако средняя выручка на календарный день выросла с 1513094.68 до 1553554.82. Аналогичная ситуация наблюдается при сравнении мая и июня: месячная выручка снизилась с 60.76 млн до 60.16 млн, а средняя дневная выросла с 1960121.68 до 2005205.10 Следовательно, снижение месячной суммы само по себе не свидетельствует об ухудшении динамики, необходимо учитывать продолжительность периода – 31 день в январе и мае, 28 дней в феврале и 30 дней в июне

После анализа помесячной динамики я выбираю два периода для подробного сравнения: январь 2026 года – базисный период и март – как отчетный (оба месяца содержат 31 календарный день, и проверка не выявила в них дней без записей о попытках оплаты, поэтому различие в продолжительности месяцев не будет влиять на сравнение их суммарной выручки)
В предварительной сводке средняя сумма успешного платежа увеличилась с 274.84 в январе до 324.83 рублей в марте, поэтому мне для дальнейшего анализа важно узнать, как изменилось количество успешных оплат и какой вклад каждый из этих факторов внес в изменение выручки 


In [ ]:
comparison = monthly.loc[
    ["2026-01", "2026-03"],
    ["revenue", "n_payments", "avg_payment", "calendar_days"]].copy()

comparison

На основе таблицы comparison рассчитываю изменение трех показателей между январем и мартом: выручки, количества успешных оплат и средней суммы платежа

In [ ]:
metrics = ["revenue", "n_payments", "avg_payment"]

january = comparison.loc["2026-01", metrics]
march = comparison.loc["2026-03", metrics]

changes = pd.DataFrame({
    "january": january,
    "march": march,
    "delta": march - january,
    "change_pct": (march - january) / january * 100})

changes.round(2)

С января по март 2026 года выручка от успешных оплат выросла с 46,91 до 58,23 млн рублей – на 11,32 млн рублей (24,14%). Количество успешных оплат увеличилось на 8 593(5,04%), а средняя сумма успешного платежа – на 49,99 рублй (18,19%). Получается, что одновременно выросли оба показателя, от которых зависит выручка. Далее я определю, в каких способах оплаты сосредоточен ее прирост, а затем рассчитаю вклад изменения количества оплат и средней суммы платежа

In [ ]:
#сравниваю выручку по способам оплаты
payments_comparison = success[
    success["month"].isin(["2026-01", "2026-03"])].copy()

revenue_by_method = payments_comparison.pivot_table(
    index="payment_method",
    columns="month",
    values="amount",
    aggfunc="sum",
    fill_value=0)

revenue_by_method["delta"] = (revenue_by_method["2026-03"] - revenue_by_method["2026-01"])

revenue_by_method = revenue_by_method.sort_values("delta", ascending=False)

revenue_by_method.round(2)

In [ ]:
#проверяю, что разложение изменения выручки по способам оплаты согласуется с общим результатом
total_delta = (comparison.loc["2026-03", "revenue"] - comparison.loc["2026-01", "revenue"])

method_delta = revenue_by_method["delta"].sum()

assert abs(method_delta - total_delta) < 0.01, \
    "Сумма приростов по способам оплаты не совпадает с общим приростом"

print("Расхождение:", round(method_delta - total_delta, 2), "₽")

revenue_by_method["contribution_pct"] = (revenue_by_method["delta"] / total_delta * 100)

revenue_by_method.round(2)

Выручка увеличилась по всем четырем способам оплаты. Наибольший вклад в общий прирост внесли карты – 5,88 млн рублей(51,94%), и СБП – 3,54 млн рублей. Вместе на них приходится 83,23% прироста. Вклад электронного кошелька составил 1,23 млн рублей, а внутреннего баланса – 0,67 млн рублей, это показывает, где сосредоточен прирост, однако еще не позволяет определить, какой способ оплаты имеет наибольший потенциал дальнейшего роста

In [ ]:
# показатели базисного периода — января
n_january = comparison.loc["2026-01", "n_payments"]
avg_january = comparison.loc["2026-01", "avg_payment"]
revenue_january = comparison.loc["2026-01", "revenue"]

# показатели отчётного периода — марта
n_march = comparison.loc["2026-03", "n_payments"]
revenue_march = comparison.loc["2026-03", "revenue"]

#меняю количество оплат, сохраняя январскую среднюю сумму
revenue_intermediate = n_march * avg_january

#вклады двух факторов
effect_n_payments = revenue_intermediate - revenue_january
effect_avg_payment = revenue_march - revenue_intermediate

factor_effects = pd.DataFrame({
    "factor": ["Количество успешных оплат", "Средняя сумма платежа"],
    "effect_rub": [effect_n_payments, effect_avg_payment]})

factor_effects

In [ ]:
#проверяю, что сумма вкладов двух факторов совпадает с общим приростом выручки между январем и мартом
#затем рассчитываю долю каждого фактора в приросте, чтобы сравнить их вклад
effects_sum = factor_effects["effect_rub"].sum()

assert abs(effects_sum - total_delta) < 0.01, \
    "Сумма вкладов факторов не совпадает с общим приростом"

print("Расхождение:", round(effects_sum - total_delta, 2), "₽")

factor_effects["contribution_pct"] = (
    factor_effects["effect_rub"] / total_delta * 100)

factor_effects.style.format({
    "effect_rub": "{:,.2f}",
    "contribution_pct": "{:.2f}"})

При выбранном порядке подстановок основная часть прироста выручки приходится на изменение средней суммы успешного платежа. Хочу отметить, что это разложение еще не объясняет почему средний платеж вырос, для этого я далее изучу тарифы и структуру оплат по ним 

In [ ]:
#сопоставялю платежи с тарифами и проверяю структуру таблиц подписок и тарифов, 
#чтобы определить поля для объединения данных и выяснить, как в базе хранится информация о тарифах и их ценах

table_columns = q("""
    select
        table_name,
        column_name,
        data_type
    from information_schema.columns
    where table_schema = 'prime'
      and table_name in ('subscriptions', 'tariffs')
    order by table_name, ordinal_position""")

table_columns

In [ ]:
tariffs = q("""
    select
        tariff_id,
        tariff_name,
        monthly_fee,
        valid_from,
        valid_to
    from prime.tariffs
    order by tariff_id, valid_from""")

tariffs


В справочнике обнаружены пересекающиеся периоды действия записей для тарифа «Расширенный»: в марте одновременно указаны цены 429 и 449 рублей, поэтому объединение платежей со всеми версиями тарифа может привести к дублированию строк и завышению выручки. Поэтому далее для сравнения тарифов я сначала сопоставлю платежи с идентификаторами тарифов через таблицу подписок, а среднюю сумму платежа буду рассчитывать по фактически оплаченным суммам, а неоднозначность справочника учту при интерпретации результатов


In [ ]:
print("Строк в pay:", len(pay))
print("Успешных оплат:", len(success))

Для оценки возможных направлений роста выручки я изучаю статусы платежей по способам оплаты за январь и март 2026 года. Использую таблицу pay, содержащую платежи всех статусовв. Сначала подсчитываю количество записей каждого статуса для каждого способа оплаты отдельно по месяцам, что позволит далее сравнить доли успешных и неуспешных платежей

In [ ]:
# Оставляем платежи за два сравниваемых месяца
pay_methods = pay[
    pay["paid_at"].dt.strftime("%Y-%m").isin(["2026-01", "2026-03"])].copy()

pay_methods["month"] = pay_methods["paid_at"].dt.strftime("%Y-%m")

# Считаем записи каждого статуса
status_by_method = pay_methods.pivot_table(
    index=["month", "payment_method"],
    columns="status",
    values="payment_id",
    aggfunc="count",
    fill_value=0)

status_by_method

In [ ]:
method_rates = status_by_method.copy()

# Общее количество платежных записей в каждой группе
method_rates["n_total"] = method_rates[["failed", "refunded", "success"]].sum(axis=1)

# Доли статусов в процентах
method_rates["success_pct"] = (method_rates["success"] / method_rates["n_total"] * 100)

method_rates["failed_pct"] = (method_rates["failed"] / method_rates["n_total"] * 100)

method_rates["refunded_pct"] = (method_rates["refunded"] / method_rates["n_total"] * 100)

method_rates[["n_total", "success_pct", "failed_pct", "refunded_pct"]].round(2)

В январе и марте доля успешных платежей у всех способов оплаты превышает 95%. В марте еще разница между максимальной и минимальной долями составляет около 0.20 процентного пункта. Наибольшее количество неуспешных платежей приходится на карты – 3193 записи, однако доля неуспешных платежей у них самая низкая среди рассматриваемых способов – 3,26%. Следовательно, большое количество неуспешных оплат картами связано прежде всего с масштабом их использования, при этом полученные результаты не дают достаточных оснований рекомендовать перевод клиентов на другой способ оплаты и для поиска возможностей роста необходимо дополнительно изучить причины отказов и успешность повторных попыток

In [ ]:
#сравнивню изменение долей между январем и мартом
rate_columns = ["success_pct", "failed_pct", "refunded_pct"]

rates_january = method_rates.loc["2026-01", rate_columns]
rates_march = method_rates.loc["2026-03", rate_columns]

rates_change = rates_march - rates_january

rates_change = rates_change.rename(columns={
    "success_pct": "success_change_pp",
    "failed_pct": "failed_change_pp",
    "refunded_pct": "refunded_change_pp"})

rates_change.round(2)

Заметного по величине ухудшения структуры статусов не наблюдается. Эти показатели описывают статусы записей в выгрузке, но не позволяют установить причины отказов или возвратов

In [ ]:
import matplotlib.pyplot as plt

failed_for_plot = method_rates["failed_pct"].unstack("month")
failed_for_plot = failed_for_plot[["2026-01", "2026-03"]]

ax = failed_for_plot.plot(
    kind="bar",
    figsize=(9, 5),
    color=["#B8BEC8", "#4472C4"],
    rot=0)

ax.set_title("Доли неуспешных платежей близки у всех способов оплаты")
ax.set_xlabel("Способ оплаты")
ax.set_ylabel("Доля записей со статусом failed, %")
ax.set_ylim(0, 4.2)

ax.legend(
    ["Январь 2026", "Март 2026"],
    title="Период",
    loc="upper left",
    bbox_to_anchor=(1.02, 1))

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()

In [ ]:
#График по вкладу двух факторов в прирост выручки 
# Переводим рубли в миллионы рублей только для графика
effects_mln = factor_effects["effect_rub"] / 1_000_000

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.barh(
    factor_effects["factor"],
    effects_mln,
    color=["#B8BEC8", "#4472C4"])

ax.bar_label(bars, fmt="%.2f", padding=5)

ax.set_title(
    "Основная часть прироста приходится на среднюю сумму платежа\n"
    "Январь → март 2026 года")
ax.set_xlabel("Вклад в прирост выручки, млн ₽")
ax.set_xlim(0, effects_mln.max() * 1.2)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

Для сравнения месяцев разной продолжительности я строю график средней выручки на календарный день. Показатель рассчитывается как сумма успешных платежей за месяц, деленная на количество календарных дней. Апрель исключаю из сравнения из-за отсутствия записей за 14 и 15 апреля (на графике оставляю разрыв, не восстанавливая значение предположениями)

In [ ]:
# Возвращаем все месяцы на ось, оставляя апрель без значения
daily_revenue_plot = monthly_comparison["revenue_per_day"].reindex(
    monthly.index)

# Для удобства отображаем миллионы рублей
daily_revenue_mln = daily_revenue_plot / 1_000_000

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    daily_revenue_mln.index,
    daily_revenue_mln,
    marker="o",
    linewidth=2,
    color="#4472C4")

ax.set_title("Средняя дневная выручка растет в сравниваемых месяцах")
ax.set_xlabel("Месяц 2026 года")
ax.set_ylabel("Выручка на календарный день, млн ₽")
ax.set_ylim(0, daily_revenue_mln.max() * 1.2)

for month, value in daily_revenue_mln.dropna().items():
    ax.annotate(
        f"{value:.2f}",
        xy=(month, value),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center")

ax.text(
    0.5, 0.05,
    "Апрель исключен: отсутствуют записи за 14–15 апреля",
    transform=ax.transAxes,
    ha="center",
    fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

backup_dir = Path("project1_backup")
backup_dir.mkdir(exist_ok=True)

pay.to_csv(
    backup_dir / "payments_2026_h1.csv",
    index=False)

tariffs.to_csv(
    backup_dir / "tariffs.csv",
    index=False)

print("Копии сохранены в:", backup_dir.resolve())

In [ ]:
subscriptions = q("""
    select
        subscription_id,
        tariff_id
    from prime.subscriptions
""")

print("Количество строк:", len(subscriptions))
print(
    "Повторяющиеся subscription_id:",
    subscriptions["subscription_id"].duplicated().sum())
print("Пропуски по столбцам:")
print(subscriptions.isna().sum())

subscriptions.head()

In [ ]:
subscriptions.to_csv("project1_backup/subscriptions.csv", index=False)

Далее я буду объединять успешные платежи за январь и март с таблицей подписок subscriptions_id, чтобы каждому платежу сопоставить индекатор тарифа 

In [ ]:
payments_with_tariff = payments_comparison.merge(
    subscriptions,
    on="subscription_id",
    how="left",
    validate="many_to_one")

#проверка, что платежи не потерялись и не размножились
assert len(payments_with_tariff) == len(payments_comparison), \
    "После объединения изменилось количество строк"

#проверка сохранение суммы выручки
revenue_difference = (payments_with_tariff["amount"].sum() - payments_comparison["amount"].sum())

assert abs(revenue_difference) < 0.01, \
    "После объединения изменилась сумма выручки"

#для всех ли платежей найден тариф
missing_tariffs = payments_with_tariff["tariff_id"].isna().sum()

print("Количество платежей:", len(payments_with_tariff))
print("Расхождение выручки:", round(revenue_difference, 2), "₽")
print("Платежей без тарифа:", missing_tariffs)

assert missing_tariffs == 0, "Для части платежей не найден тариф"

payments_with_tariff.head()

После объединения количество платежей и сумма выручки сохранились, платежей без идентификатора тарифа нет. Далее я группирую успешные платежи по месяцу и тарифу и рассчитываю выручку, количество оплат и среднюю сумму платежа. Дополнительно проверяю минимальную, максимальную суммы и количество различных сумм внутри каждой группы, чтобы выяснить, все ли платежи одного тарифа в пределах месяца имеют одинаковую сумму

In [ ]:
tariff_summary = payments_with_tariff.groupby(
    ["month", "tariff_id"]).agg(
    revenue=("amount", "sum"),
    n_payments=("payment_id", "size"),
    avg_payment=("amount", "mean"),
    min_payment=("amount", "min"),
    max_payment=("amount", "max"),
    n_unique_amounts=("amount", "nunique"))

tariff_summary.round(2)

В январе все успешные платежи по «Базовому» тарифу составили 199 рублей, по «Расширенному» – 399 рублей. В марте суммы составили соответственно 249 и 449 рублей. Таким образом, внутри каждого тарифа сумма платежа увеличилась на 50 рублей, при этом общая средняя сумма платежа выросла примерно на 49.99 рубля. Далее я проверю, как изменилось соотношение тарифов в количестве успешных оплат, поскольку оно также влияет на общую среднюю сумму

In [ ]:
# Общее количество успешных оплат за каждый месяц
monthly_payment_counts = tariff_summary.groupby(level="month")["n_payments"].transform("sum")

# Доля каждого тарифа в количестве оплат своего месяца
tariff_summary["payment_share_pct"] = (tariff_summary["n_payments"] / monthly_payment_counts * 100)

tariff_summary[["n_payments", "avg_payment", "payment_share_pct"]].round(4)

Cоотношение почти не изменилось. Таким образом, увеличение общей средней суммы платежа связано преимущественно с ростом суммы оплаты внутри обоих тарифов на 50 рублей. Небольшое изменение структуры оказывает противоположное влияние.

In [ ]:
#выразим эти два вклада в рублях среднего платежа
tariffs_january = tariff_summary.loc["2026-01"]
tariffs_march = tariff_summary.loc["2026-03"]

# Доли переводим из процентов в части единицы
shares_january = tariffs_january["payment_share_pct"] / 100
shares_march = tariffs_march["payment_share_pct"] / 100

# Средневзвешенные суммы платежа
avg_start = (tariffs_january["avg_payment"] * shares_january).sum()

avg_intermediate = (tariffs_march["avg_payment"] * shares_january).sum()

avg_end = (tariffs_march["avg_payment"] * shares_march).sum()

avg_effects = pd.DataFrame({
    "factor": [
        "Суммы платежей внутри тарифов",
        "Структура оплат по тарифам"],
    "effect_rub_per_payment": [
        avg_intermediate - avg_start,
        avg_end - avg_intermediate]})

# Сверяем с изменением среднего платежа из общей сводки
actual_avg_change = (
    comparison.loc["2026-03", "avg_payment"]
    - comparison.loc["2026-01", "avg_payment"])

assert abs(
    avg_effects["effect_rub_per_payment"].sum() - actual_avg_change) < 0.000001, "Разложение не совпало с изменением среднего платежа"

avg_effects.round(4)

Рост среднего платежа практически полностью объясняется увеличением суммы оплаты внутри обоих тарифов, так как есть незначительное снижение доли «Расширенного» тарифа, которое немного уменьшило этот рост

Итак, далее уточню разложение общего прироста выручки: я сохраняю рассчитанный вклад количества успешных оплат, а вклад среднего платежа разделяю на изменение сумм внутри тарифов и изменение структуры оплат. Для перевода вкладов из рублей на один платеж в рубли выручки умножаю их на мартовское количество оплат.

In [ ]:
n_march = comparison.loc["2026-03", "n_payments"]

effect_tariff_amounts = (avg_intermediate - avg_start) * n_march
effect_tariff_structure = (avg_end - avg_intermediate) * n_march

revenue_factors = pd.DataFrame({
    "factor": [
        "Количество успешных оплат",
        "Суммы платежей внутри тарифов",
        "Структура оплат по тарифам"],
    "effect_rub": [
        effect_n_payments,
        effect_tariff_amounts,
        effect_tariff_structure]})

#все три вклада должны дать общий прирост выручки
difference = revenue_factors["effect_rub"].sum() - total_delta

assert abs(difference) < 0.01, \
    "Сумма вкладов не совпадает с общим приростом"

print("Расхождение:", round(difference, 2), "₽")

revenue_factors.style.format({
    "effect_rub": "{:,.2f}"})

Далее отвечу на вопрос: какой тариф внес больший вклад

In [ ]:
revenue_by_tariff = tariff_summary["revenue"].unstack("month")

revenue_by_tariff["delta"] = (revenue_by_tariff["2026-03"] - revenue_by_tariff["2026-01"])

revenue_by_tariff["contribution_pct"] = (revenue_by_tariff["delta"] / total_delta * 100)

assert abs(revenue_by_tariff["delta"].sum() - total_delta) < 0.01, "Прирост по тарифам не совпадает с общим приростом"

revenue_by_tariff.style.format({
    "2026-01": "{:,.2f}",
    "2026-03": "{:,.2f}",
    "delta": "{:,.2f}",
    "contribution_pct": "{:.2f}"})

Выручка «Базового» тарифа увеличилась на 6,63 млн рублей, что составляет 58,53% общего прироста, а выручка «Расширенного» выросла на 4,70 млн рублей – 41,47% прироста. Хотя «Расширенный» приносит больше выручки в обоих месяцах, больший абсолютный прирост наблюдается у «Базового». Это согласуется с более высоким количеством оплат «Базового» при одинаковом увеличении суммы платежа на 50 рублей в обоих тарифах

На диаграмме показываю абсолютный прирост выручки по каждому тарифу между январем и мартом

In [ ]:
# Названия соответствуют идентификаторам в справочнике тарифов
tariff_growth = revenue_by_tariff["delta"].rename(index={
    1: "Базовый",
    2: "Расширенный"}) / 1_000_000

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.barh(
    tariff_growth.index,
    tariff_growth.values,
    color=["#4472C4", "#B8BEC8"])

ax.bar_label(bars, fmt="%.2f", padding=5)

ax.set_title(
    "Базовый тариф обеспечил большую часть прироста выручки\n"
    "Январь → март 2026 года")
ax.set_xlabel("Прирост выручки, млн ₽")
ax.set_xlim(0, tariff_growth.max() * 1.2)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Увеличение доли на 1 процентный пункт
share_shift = 0.01

n_march = comparison.loc["2026-03", "n_payments"]

basic_amount = tariff_summary.loc[("2026-03", 1), "avg_payment"]
extended_amount = tariff_summary.loc[("2026-03", 2), "avg_payment"]

# Разница сумм успешной оплаты двух тарифов
amount_gap = extended_amount - basic_amount

# Сценарный прирост выручки
scenario_revenue_gain = n_march * share_shift * amount_gap

print("Разница сумм платежей:", round(amount_gap, 2), "₽")
print(
    "Прирост выручки при сдвиге доли на 1 п. п.:",
    round(scenario_revenue_gain, 2),"₽")

Итак, в работе я проанализировала выручку от успешных оплат подписки «Прайм» за первое полугодие 2026 года. Отдельно стоит сказать, что ответ на поставленный вопрос исполнен и поставленные задачи выполнены. Теперь об анализе,  для подробного сравнения выбрала январь и март, так как оба месяца содержат 31 календарный день, а март стал месяцем резкого изменения. Еще проверка показала, что проверка не выявила в них дней без записей о платежах, что дополнительно дало мне использовать эти месяцы. 


Теперь об анализе относительно выручки, с января по март выручка выросла с 46,91 до 58,23 млн рублей, то есть на 11,32 млн или 24,14%. Количество успешных оплат при этом тоже увеличилось на 5,04%, а средняя сумма успешного платежа – на 18,19% (с 274,84 до 324,83 рублей). Стоит отметить, что при анализе полугодия я также учла продолжительность месяцев. В феврале и июне месячная выручка была ниже, чем в предшествующем месяце, однако выручка на календарный день выросла, поэтому уменьшение месячной суммы само по себе не означает ухудшения динамики. Для разложения изменения выручки я использовала метод цепных подстановок в порядке от количество успешных оплат к сумме платежей внутри тарифов и далее к структура оплат по тарифам.  При таком порядке увеличение количества успешных оплат дало вклад 2,36 млн рублей, изменение сумм платежей внутри тарифов – 8,96 млн рублей, а изменение структуры оплат – −1,51 тыс. рублей. Сумма вкладов совпала с общим приростом выручки, следовательно, основной вклад в прирост связан с увеличением сумм платежей внутри тарифов, а влияние изменения структуры оказалось незначительным.


Чуть подробнее в цифрах, так, основная часть прироста связана с увеличением сумм платежей внутри тарифов. В январе успешные платежи по «Базовому» составляли 199 рублей, а в марте – 249. По «Расширенному» – 399 и 449 соответственно. Таким образом, сумма платежа выросла на 50 рублей в обоих тарифах, а соотношение количества оплат между тарифами практически не изменилось. В разрезе тарифов на «Базовый» приходится 6,63 миллионов или 58,53% общего прироста, а на «Расширенный» – 4,70 млн или 41,47%. Больший вклад «Базового» согласуется с тем, что при одинаковом увеличении суммы платежа на него приходится больше успешных оплат и при этом абсолютная выручка «Расширенного» в марте остается выше (30,52 миллиона против 27,71 миллиона). В разрезе способов оплаты было получено, что основной вклад в прирост дали карты – 51,94% и СБП – 31,29%. Доли записей со статусом “failed” близки у всех способов оплаты: 3,24-3,43% в январе и 3,26-3,41% в марте, но эти результаты не дают оснований объяснять прирост выручки заметным улучшением успешности платежей.Такие разложения по тарифам, способам оплаты и факторам описывают один и тот же прирост с разных сторон, поэтому их результаты нельзя складывать между собой.


Предложения: 
1) Проверить предложение перехода на тариф «Расширенный». Почему этот пункт предложен, потому что из  полученных данных стало известно, что при увеличении доли тарифа в количестве успешных оплат на 1 процентный пункт за счет «Базового» дополнительная месячная выручка составила бы 358516 рублей при неизменных мартовских количестве оплат и суммах платежей. Это условный сценарий, а не прогноз, но предложение следует проверить в эксперименте с контрольной группой, учитывая дополнительную выручку, затраты и отмены подписок.

2) Еще одно предложение – проверить удержание после увеличения стоимости, так как на «Базовый» приходится около 62,08% успешных оплат марта, поэтому сохранение этого объема существенно для выручки. Поэтому можно сравнить продления и отмены подписок до и после изменения стоимости отдельно по тарифам, потому что полученные расчеты не показывают ухудшения удержания, но и не позволяют его исключить, так как рост числа оплат может скрывать уход одних клиентов и приток других. 

3) Последнее – исследовать причины неуспешных оплат прежде всего по картам и СБП. отвечая на вопрос почему стоит сказать, что на эти способы приходится наибольший объем платежей, поэтому они представляют интерес для поиска возможностей улучшения. Однако близость долей failed не позволяет считать какой-либо способ явно проблемным и для оценки возможного эффекта нужно связать неуспешные попытки с последующими успешными оплатами.


Ограничения анализа:
- Выручка в работе определена как сумма платежей со статусом “success”. Тут это как показатель денежных поступлений по выбранному правилу, а не бухгалтерская выручка, распределенная по сроку оказания услуг
- За 14-15 апреля в выгрузке отсутствуют записи о платежах всех статусов. Причина неизвестна, поэтому апрель исключен из основного сравнения полных месяцев
- В справочнике тарифов обнаружены пересекающиеся сроки действия записей, поэтому для расчетов использовались фактические суммы платежей, а тариф определялся через подписку.
- Факторное разложение показывает арифметические составляющие изменения выручки и они не доказывают причинное влияние факторов на поведение клиентов, потому что величины вкладов зависят от выбранного порядка подстановок
- Количество платежей не равно количеству клиентов (неуспешная попытка могла завершиться повторной успешной оплатой, поэтому сумму неуспешных платежей нельзя автоматически считать потерянной выручкой).
- Анализ января и марта не позволяет оценить долгосрочную реакцию клиентов на увеличение стоимости подписки, поэтому для выбора наиболее выгодного направления роста лучше использовать дополнительные данные об удержании клиентов, затратах и результатах экспериментов.

